# Colab / Kaggle runner

Runs the training pipeline of
[chest-xray-pneumonia-transfer-learning](https://github.com/emirhantash/chest-xray-pneumonia-transfer-learning)
on a notebook GPU. Run the cells from top to bottom.

- **Colab:** *Runtime > Change runtime type > GPU*. Outputs go to Google Drive, so an
  interrupted run can be resumed in a new session.
- **Kaggle:** *Settings > Accelerator > GPU*, and *Add Input* the dataset
  `paultimothymooney/chest-xray-pneumonia`. Outputs go to `/kaggle/working/outputs`
  (kept only when the notebook version is saved).

The original test set is **not** used here; it is reserved for the final evaluation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").is_dir()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
print(f"Colab: {IN_COLAB}, Kaggle: {IN_KAGGLE}, work dir: {WORK_DIR}")

## 1. Get the code

In [ ]:
REPO = "emirhantash/chest-xray-pneumonia-transfer-learning"
REPO_DIR = WORK_DIR / REPO.split("/")[1]
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Install the dependencies

PyTorch is preinstalled on Colab and Kaggle. scikit-learn ≥ 1.8 is required by
`requirements.txt`; the train/val split itself is read from `splits/patient_seed42.csv`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sklearn
import torch

print(f"torch {torch.__version__}, scikit-learn {sklearn.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

## 3. Get the data

Kaggle: the dataset added as an input is found under `/kaggle/input`.
Colab: the dataset is downloaded with `kagglehub`; this may need Kaggle API credentials
(`KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets).

In [ ]:
def find_data_root(base: Path) -> Path:
    """The folder that directly contains train/ and test/ (the archive nests a copy)."""
    for train_dir in sorted(base.rglob("train")):
        root = train_dir.parent
        if (root / "test").is_dir() and "__MACOSX" not in root.parts:
            return root
    raise FileNotFoundError(f"No folder with train/ and test/ under {base}")


if IN_KAGGLE:
    DATA_ROOT = find_data_root(Path("/kaggle/input"))
else:
    import kagglehub

    DATA_ROOT = find_data_root(
        Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
    )
print("data root:", DATA_ROOT)

## 4. Output folder

On Colab, Google Drive is mounted so that checkpoints survive a disconnect.

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/chest-xray-pneumonia/outputs")
else:
    OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs:", OUTPUT_DIR)

## 5. Sanity check

Should end with `ALL CHECKS PASSED` and report that the recomputed split is identical to
the split file.

In [ ]:
!python scripts/check_data.py --config configs/baseline.yaml --data-root "{DATA_ROOT}"

## 6. Train

Rerunning this cell after a disconnect resumes from the last completed epoch
(`--resume` is added automatically when a checkpoint exists). Each checkpoint is about
435 MB.

In [ ]:
CONFIG = "configs/baseline.yaml"
RUN_NAME = "baseline"
train_args = f'--config {CONFIG} --data-root "{DATA_ROOT}" --output-dir "{OUTPUT_DIR}"'
train_args += f" --run-name {RUN_NAME}"
if (OUTPUT_DIR / RUN_NAME / "last.pt").exists():
    train_args += " --resume"
!python scripts/train.py {train_args}

## 7. Results (validation set)

In [ ]:
import json

from IPython.display import Image, display

run_dir = OUTPUT_DIR / RUN_NAME
report = json.loads((run_dir / "val_metrics.json").read_text())
print(json.dumps(report, indent=2))
for name in ("curves.png", "confusion_val.png", "roc_val.png"):
    display(Image(filename=str(run_dir / name)))

## 8. Hyperparameter optimization (Phase 3)

One study = one method and one sampler seed, 30 completed trials (`configs/hpo.yaml`),
about 1.5–2 hours per study on a Kaggle T4 or an RTX 3050 Ti (measured). Trials run one after another; a study that is
interrupted continues from the last completed trial when this cell is run again (the
interrupted trial is repeated with the same parameters).

The study database is written to local disk and copied to `HPO_BACKUP_DIR` after every
trial (Google Drive on Colab; on Kaggle, save a notebook version to keep the outputs).
Skip sections 5–7 when only running HPO.

In [ ]:
METHOD = "gp"  # "gp" (Bayesian optimization) or "random" (Random Search)
SEED = 0  # sampler seed: 0, 1, 2
HPO_DIR = WORK_DIR / "hpo"  # local disk
HPO_BACKUP_DIR = OUTPUT_DIR / "hpo"  # Drive on Colab
hpo_args = f'--method {METHOD} --seed {SEED} --data-root "{DATA_ROOT}" --output-dir "{HPO_DIR}"'
if IN_COLAB:
    hpo_args += f' --backup-dir "{HPO_BACKUP_DIR}"'
!python scripts/hpo.py {hpo_args}

Retrain the best configuration of a finished study with the full training pipeline
(checkpoints, curves, predictions):

In [ ]:
best_config = HPO_DIR / f"{METHOD}_seed{SEED}" / "best_config.yaml"
best_args = f'--config "{best_config}" --data-root "{DATA_ROOT}" --output-dir "{OUTPUT_DIR}"'
!python scripts/train.py {best_args}